In [1]:
from pathlib import Path
from dataclasses import dataclass
from functools import cached_property

# from bisect import bisect_left, bisect_right
import numpy as np
import numpy.typing as npt
from scipy.signal import correlate

from mbloodmoon.mask import _bisect_interval
from mbloodmoon.mask import _fold
from mbloodmoon.types import BinsRectangular, UpscaleFactor
from mbloodmoon.io import MaskDataLoader
import mbloodmoon as bm


def _bins(
    start: float,
    stop: float,
    px_size: float,
    upscaling: int,
) -> npt.NDArray:
    """
    Returns equally spaced points between start and stop, included.
    The input `start`, `stop` and `px_size` must have same dimension.

    Args:
        start (float): Start point.
        stop (float): Stop point.
        px_size (float): Size of the pixels.
        upscaling (int): Upscaling factor.

    Returns:
        output (npt.NDArray): Bin edges array.
    """
    return np.linspace(start, stop, int((stop - start) * upscaling / px_size) + 1)


def _enlarge(
    m: npt.NDArray,
    upscale_f: UpscaleFactor,
) -> npt.NDArray:
    """
    Oversamples a 2D array by repeating elements along the axes.

    Args:
        m (npt.NDArray): Input 2D array.
        upscale_f (UpscaleFactor): Upscaling factors.

    Returns:
        output (npt.NDArray): Oversampled array.

    Notes:
        - the total sum is not conserved.
    """    
    for i, f in enumerate(upscale_f[::-1]):
        m = np.repeat(m, f, axis=i)
    return m





@dataclass(frozen=True)
class CodedMaskCamera:
    """
    Dataclass containing a coded mask camera system.

    Handles mask pattern, detector geometry, and related calculations for coded mask imaging.

    Args:
        mdl: Mask data loader object containing mask and detector specifications.
        upscale_f: Tuple of upscaling factors for x and y dimensions.
    """

    mdl: MaskDataLoader
    upscale_f: UpscaleFactor

    def _bins_mask(
        self,
        upscale_f: UpscaleFactor,
    ) -> BinsRectangular:
        """Generate binning structure for mask with given upscale factors."""
        return BinsRectangular(
            _bins(self.mdl["mask_minx"], self.mdl["mask_maxx"], self.mdl["mask_deltax"], upscale_f.x),
            _bins(self.mdl["mask_miny"], self.mdl["mask_maxy"], self.mdl["mask_deltay"], upscale_f.y),
        )
    
    # def _bins_detector(
    #     self,
    #     upscale_f: UpscaleFactor,
    # ) -> BinsRectangular:
    #     """Generate binning structure for detector with given upscale factors."""
    #     mask_bins = self._bins_mask(upscale_f)
    #     xmin, xmax = _bisect_interval(mask_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
    #     ymin, ymax = _bisect_interval(mask_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
    #     return BinsRectangular(
    #         mask_bins.x[xmin : xmax + 1],
    #         mask_bins.y[ymin : ymax + 1],
    #     )
    
    def _bins_detector(
        self,
        upscale_f: UpscaleFactor,
    ) -> BinsRectangular:
        """Generate binning structure for detector with given upscale factors."""
        base_mask_bins = self._bins_mask(UpscaleFactor(1, 1))
        mask_bins = self.bins_mask
        xmin, xmax = _bisect_interval(base_mask_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
        ymin, ymax = _bisect_interval(base_mask_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
        return BinsRectangular(
            mask_bins.x[xmin * upscale_f.x : xmax * upscale_f.x + 1],
            mask_bins.y[ymin * upscale_f.y : ymax * upscale_f.y + 1],
        )
    
    def _bins_sky(
        self,
        upscale_f: UpscaleFactor,
    ) -> BinsRectangular:
        """Binning structure for the reconstructed sky image."""
        m_bins, d_bins = self._bins_mask(upscale_f), self._bins_detector(upscale_f)
        sy, sx = self.sky_shape
        xstep, ystep = (
            m_bins.x[1] - m_bins.x[0],
            m_bins.y[1] - m_bins.y[0],
        )
        return BinsRectangular(
            np.linspace(m_bins.x[0] + d_bins.x[0] + xstep, m_bins.x[-1] + d_bins.x[-1], sx + 1),
            np.linspace(m_bins.y[0] + d_bins.y[0] + ystep, m_bins.y[-1] + d_bins.y[-1], sy + 1),
        )

    @property
    def specs(self) -> dict:
        """Returns a dictionary of mask parameters useful for image reconstruction."""
        return self.mdl.specs
    
    @cached_property
    def bins_mask(self) -> BinsRectangular:
        """Binning structure for the mask pattern."""
        return self._bins_mask(self.upscale_f)

    @cached_property
    def bins_detector(self) -> BinsRectangular:
        """Binning structure for the detector."""
        return self._bins_detector(self.upscale_f)
    
    @cached_property
    def bins_sky(self) -> BinsRectangular:
        """Returns bins for the sky-shift domain."""
        return self._bins_sky(self.upscale_f)
    
    @cached_property
    def mask(self) -> npt.NDArray:
        """2D array representing the coded mask pattern."""
        base = _fold(self.mdl.mask, self._bins_mask(UpscaleFactor(1, 1))).astype(int)
        return _enlarge(base, self.upscale_f)

    @cached_property
    def decoder(self) -> npt.NDArray:
        """2D array representing the mask pattern used for decoding."""
        base = _fold(self.mdl.decoder, self._bins_mask(UpscaleFactor(1, 1)))
        return _enlarge(base, self.upscale_f)
    
    # @cached_property
    # def bulk(self) -> npt.NDArray:
    #     """2D array representing the bulk (sensitivity) array of the mask."""
    #     framed_bulk = _fold(self.mdl.bulk, self._bins_mask(UpscaleFactor(1, 1)))
    #     framed_bulk[~np.isclose(framed_bulk, np.zeros_like(framed_bulk))] = 1
    #     bins = self._bins_mask(self.upscale_f)
    #     xmin, xmax = _bisect_interval(bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
    #     ymin, ymax = _bisect_interval(bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
    #     return _enlarge(framed_bulk, self.upscale_f)[ymin : ymax, xmin : xmax]

    @cached_property
    def bulk(self) -> npt.NDArray:
        """2D array representing the bulk (sensitivity) array of the mask."""
        base_bins = self._bins_mask(UpscaleFactor(1, 1))
        framed_bulk = _fold(self.mdl.bulk, base_bins)
        framed_bulk[~np.isclose(framed_bulk, np.zeros_like(framed_bulk))] = 1
        print(
            framed_bulk.shape,
            self.mask_shape,
            np.unique(framed_bulk),
            framed_bulk.sum() * np.prod(self.upscale_f),
        )

        xmin, xmax = _bisect_interval(base_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
        ymin, ymax = _bisect_interval(base_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
        detector = _enlarge(framed_bulk[ymin : ymax, xmin : xmax], self.upscale_f)
        print(
            detector.shape,
            self.detector_shape,
            np.unique(detector),
            detector.sum(),
        )

        n_zero_resp_pxs_x = int(np.abs((base_bins.x[xmin] - self.mdl["detector_minx"]) * self.upscale_f.x / self.mdl["mask_deltax"]))
        n_zero_resp_pxs_y = int(np.abs((base_bins.y[ymin] - self.mdl["detector_miny"]) * self.upscale_f.y / self.mdl["mask_deltay"]))
        if n_zero_resp_pxs_x > 0:
            detector[:, :n_zero_resp_pxs_x] = 0; detector[:, -n_zero_resp_pxs_x:] = 0
        if n_zero_resp_pxs_y > 0:
            detector[:n_zero_resp_pxs_y, :] = 0; detector[-n_zero_resp_pxs_y:, :] = 0
            
        print(np.unique(detector), detector.sum())

        test_mask_bins = self._bins_mask(self.upscale_f)
        test_xmin, test_xmax = _bisect_interval(test_mask_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
        test_ymin, test_ymax = _bisect_interval(test_mask_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
        print(
            f"n_zero_resp_pxs_y: {n_zero_resp_pxs_y * 2}\n"
            f"delta len det bins y: {len(test_mask_bins.y[test_ymin : test_ymax + 1]) - len(self.bins_detector.y)}\n"
            f"n_zero_resp_pxs_x: {n_zero_resp_pxs_x * 2}\n"
            f"delta len det bins x: {len(test_mask_bins.x[test_xmin : test_xmax + 1]) - len(self.bins_detector.x)}\n"
        )
        return detector

    @cached_property
    def balancing(self) -> npt.NDArray:
        """2D array representing the correlation between decoder and bulk patterns."""
        return correlate(self.decoder, self.bulk, mode="full")
    
    @cached_property
    def mask_shape(self) -> tuple[int, int]:
        """Shape of the mask array (rows, columns)."""
        bins = self.bins_mask
        return len(bins.y) - 1, len(bins.x) - 1
    
    @cached_property
    def detector_shape(self) -> tuple[int, int]:
        """Shape of the detector array (rows, columns)."""
        bins = self.bins_detector
        return len(bins.y) - 1, len(bins.x) - 1
    
    @cached_property
    def sky_shape(self) -> tuple[int, int]:
        """Shape of the reconstructed sky image (rows, columns)."""
        n, m = self.mask_shape
        u, v = self.detector_shape
        return n + u - 1, m + v - 1





def codedmask(
    mask_filepath: str | Path,
    upscale_x: int = 1,
    upscale_y: int = 1,
) -> CodedMaskCamera:
    """
    An interface to CodedMaskCamera.

    Args:
        mask_filepath: a str or a path object pointing to the mask filepath.
        upscale_x: upscaling factor over the x direction.
        upscale_y: upscaling factor over the y direction.

    Returns:
        a CodedMaskCamera object.

    Raises:
        ValueError: if physical detector plane is larger than mask.
        ValueError: if upscale factors are not positive integers.
    """
    mdl = MaskDataLoader(mask_filepath)

    if not (
        # fmt: off
        mdl["detector_minx"] >= mdl["mask_minx"] and
        mdl["detector_maxx"] <= mdl["mask_maxx"] and
        mdl["detector_miny"] >= mdl["mask_miny"] and
        mdl["detector_maxy"] <= mdl["mask_maxy"]
        # fmt: on
    ):
        raise ValueError("Detector plane is larger than mask.")

    if not ((isinstance(upscale_x, int) and upscale_x > 0) and (isinstance(upscale_y, int) and upscale_y > 0)):
        raise ValueError("Upscale factors must be positive integers.")

    return CodedMaskCamera(mdl, UpscaleFactor(x=upscale_x, y=upscale_y))



# def _bins_detector(
#     self,
#     upscale_f: UpscaleFactor,
# ) -> BinsRectangular:
#     """Generate binning structure for detector with given upscale factors."""
#     base_mask_bins = self._bins_mask(UpscaleFactor(1, 1))
#     mask_bins = self.bins_mask
#     xmin, xmax = _bisect_interval(base_mask_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
#     ymin, ymax = _bisect_interval(base_mask_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
#     return BinsRectangular(
#         mask_bins.x[xmin * upscale_f.x : xmax * upscale_f.x + 1],
#         mask_bins.y[ymin * upscale_f.y : ymax * upscale_f.y + 1],
#     )

# @cached_property
# def bulk(self) -> npt.NDArray:
#     """2D array representing the bulk (sensitivity) array of the mask."""
#     base_bins = self._bins_mask(UpscaleFactor(1, 1))
#     framed_bulk = _fold(self.mdl.bulk, base_bins)
#     framed_bulk[~np.isclose(framed_bulk, np.zeros_like(framed_bulk))] = 1
#     xmin, xmax = _bisect_interval(base_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
#     ymin, ymax = _bisect_interval(base_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
#     framed_bulk[ymin : ymin + self.upscale_f.y // 2] = 0
#     framed_bulk[ymax - self.upscale_f.y // 2 : ymax] = 0
#     return (
#         _enlarge(
#             framed_bulk,
#             self.upscale_f,
#         )[
#             ymin * self.upscale_f.y : ymax * self.upscale_f.y,
#             xmin * self.upscale_f.x : xmax * self.upscale_f.x
#         ]
#     )

# test_mask_bins = self._bins_mask(self.upscale_f)
# test_xmin, test_xmax = _bisect_interval(test_mask_bins.x, self.mdl["detector_minx"], self.mdl["detector_maxx"])
# test_ymin, test_ymax = _bisect_interval(test_mask_bins.y, self.mdl["detector_miny"], self.mdl["detector_maxy"])
# print(
#     f"n_zero_resp_pxs_y: {n_zero_resp_pxs_y * 2}\n"
#     f"delta len det bins y: {len(test_mask_bins.y[test_ymin : test_ymax + 1]) - len(self.bins_detector.y)}\n"
#     f"n_zero_resp_pxs_x: {n_zero_resp_pxs_x * 2}\n"
#     f"delta len det bins x: {len(test_mask_bins.x[test_xmin : test_xmax + 1]) - len(self.bins_detector.x)}\n"
# )

In [2]:
def print_info(
    mask_path: str,
    upscale_to: int,
    base_cam: CodedMaskCamera,
    _start: int = 0,
) -> None:
    
    for ups_y, ups_x in tuple(
        (i + 1, i + 1) for i in range(_start, upscale_to)
    ):
        wfm = codedmask(mask_path, ups_x, ups_y)
        mask_bins = wfm.bins_mask
        detector_bins = wfm.bins_detector
        sky_bins = wfm.bins_sky

        print(f"############ {ups_y, ups_x} ############")
        for (idx, ax), b, up in zip(
            enumerate(("y", "x")), (1, 0), (ups_y, ups_x),
        ):
            print(
                f"## {ax.upper()} AXIS\n"
                
                f"  - Mask edges: {mask_bins[b][0], mask_bins[b][-1]} (over {wfm.specs["mask_min" + ax], wfm.specs["mask_max" + ax]})\n"
                f"  - Detector edges: {detector_bins[b][0], detector_bins[b][-1]} (over {wfm.specs["detector_min" + ax], wfm.specs["detector_max" + ax]})\n"
                f"  - Sky edges: {sky_bins[b][0], sky_bins[b][-1]}\n\n"
                
                f"  - Mask bins {ax} step: {mask_bins[b][1] - mask_bins[b][0]} (over {wfm.specs["mask_delta" + ax] / up})\n"
                f"  - Detector bins {ax} step: {detector_bins[b][1] - detector_bins[b][0]} (over {wfm.specs["mask_delta" + ax] / up})\n"
                f"  - Sky bins {ax} step: {sky_bins[b][1] - sky_bins[b][0]} (over {wfm.specs["mask_delta" + ax] / up})\n"
                f"  - Mask-Sky superimposed binning {ax}: {np.all(np.abs(mask_bins[b] - sky_bins[b][len(detector_bins[b]) // 2 - 1 : -len(detector_bins[b]) // 2 + 1]) < 1e-7)}\n\n"
                
                f"  - Up mask shape {ax} vs WFMbase: {wfm.mask_shape[idx]}/{base_cam.mask_shape[idx]} (x{wfm.mask_shape[idx]/base_cam.mask_shape[idx]})\n"
                f"  - Up detector shape {ax} vs WFMbase: {wfm.detector_shape[idx]}/{base_cam.detector_shape[idx]} (x{wfm.detector_shape[idx]/base_cam.detector_shape[idx]})\n"
                f"  - Up sky shape {ax} vs WFMbase: {wfm.sky_shape[idx]}/{(base_cam.mask_shape[idx] + base_cam.detector_shape[idx]) * up - 1} (x{wfm.sky_shape[idx]/((base_cam.mask_shape[idx] + base_cam.detector_shape[idx]) * up - 1)})\n"
                f"  - Up sky shape {ax} vs WFM: {wfm.sky_shape[idx]}/{wfm.mask_shape[idx] + wfm.detector_shape[idx] - 1} (x{wfm.sky_shape[idx]/(wfm.mask_shape[idx] + wfm.detector_shape[idx] - 1)})\n\n"
                
                f"  - Delta mask {ax}: {wfm.mask_shape[idx] - base_cam.mask_shape[idx] * up}\n"
                f"  - Delta detector {ax}: {wfm.detector_shape[idx] - base_cam.detector_shape[idx] * up}\n"
                f"  - Delta sky {ax} vs WFMbase: {wfm.sky_shape[idx] - ((base_cam.mask_shape[idx] + base_cam.detector_shape[idx]) * up - 1)}\n"
                f"  - Delta sky {ax} vs WFM: {wfm.sky_shape[idx] - (wfm.mask_shape[idx] + wfm.detector_shape[idx] - 1)}\n"
            )

        print(
            f"- Mask values: {np.unique(wfm.mask)}\n"
            f"- Decoder values: {np.unique(wfm.decoder)}\n"
            f"- Bulk values: {np.unique(wfm.bulk)}\n"
            f"- Bulk sum wrt base bulk: {wfm.bulk.sum() / base_cam.bulk.sum()}\n\n"

            f"- Mask shape WFM: {wfm.mask.shape}/{wfm.mask_shape} (x{np.array(wfm.mask.shape) / np.array(wfm.mask_shape)})\n"
            f"- Decoder shape WFM: {wfm.decoder.shape}/{wfm.mask_shape} (x{np.array(wfm.decoder.shape) / np.array(wfm.mask_shape)})\n"
            f"- Bulk shape WFM: {wfm.bulk.shape}/{wfm.detector_shape} (x{np.array(wfm.bulk.shape) / np.array(wfm.detector_shape)})\n"
            f"- Balancing shape WFM: {wfm.balancing.shape}/{wfm.sky_shape} (x{np.array(wfm.balancing.shape) / np.array(wfm.sky_shape)})\n"

            # f"Difference detector bins - alt detector bins: {wfm.bins_detector.y.size - wfm.bins_detector_alt.y.size, wfm.bins_detector.x.size - wfm.bins_detector_alt.x.size}\n"
            # f"Difference bulk - alt bulk: {np.array(wfm.bulk.shape) - np.array(wfm.bulk_alt.shape)}\n"
        )
        print("\n\n")


mask_path = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/wfm_mask.fits"
# mask_path = "/mnt/d/PhD_AASS/Coding/Images_fits/wfm_mask.fits"
upscale_to = 5
print_info(
    mask_path=mask_path,
    upscale_to=upscale_to,
    base_cam=codedmask(mask_path),
)

############ (1, 1) ############
## Y AXIS
  - Mask edges: (np.float64(-130.0), np.float64(130.0)) (over (-130.0, 130.0))
  - Detector edges: (np.float64(-76.8), np.float64(76.80000000000001)) (over (-76.5255, 76.5255))
  - Sky edges: (np.float64(-206.4), np.float64(206.8))

  - Mask bins y step: 0.4000000000000057 (over 0.4)
  - Detector bins y step: 0.3999999999999915 (over 0.4)
  - Sky bins y step: 0.4000000000000057 (over 0.4)
  - Mask-Sky superimposed binning y: True

  - Up mask shape y vs WFMbase: 650/650 (x1.0)
  - Up detector shape y vs WFMbase: 384/384 (x1.0)
  - Up sky shape y vs WFMbase: 1033/1033 (x1.0)
  - Up sky shape y vs WFM: 1033/1033 (x1.0)

  - Delta mask y: 0
  - Delta detector y: 0
  - Delta sky y vs WFMbase: 0
  - Delta sky y vs WFM: 0

## X AXIS
  - Mask edges: (np.float64(-130.0), np.float64(130.0)) (over (-130.0, 130.0))
  - Detector edges: (np.float64(-79.0), np.float64(79.0)) (over (-78.988, 78.988))
  - Sky edges: (np.float64(-208.75), np.float64(209.0))

 

In [2]:
from mbloodmoon.mask import encode, decode

def sky_upscaling(
    sky: npt.NDArray,
    base_camera: CodedMaskCamera,
    upscaled_camera: CodedMaskCamera,
) -> npt.NDArray:
    """
    Upscales the sky by redefining the binning structure, taking into
    account that in Coded Aperture Imaging applications the sky is a
    decoded image obtained by a correlation operation.

    Args:
        sky (npt.NDArray):
            Input 2D sky.
        base_camera (CodedMaskCamera):
            CodedMaskCamera instance with info on input sky binning.
        upscaled_camera (CodedMaskCamera):
            CodedMaskCamera instance with info on upscaled binning.

    Returns:
        output (npt.NDArray): Oversampled sky.
    
    Notes:
        - The detector total sum is conserved through linear interpolation.
        - This method does not perform a "standard" oversampling due to the
          way the sky is obtained (see `astropy.nndata.block_replicate()`).
    """    
    upscaling = UpscaleFactor(
        upscaled_camera.upscale_f.x - base_camera.upscale_f.x + 1,
        upscaled_camera.upscale_f.y - base_camera.upscale_f.y + 1,
    )
    detector = _enlarge(encode(base_camera, sky), upscaling) / np.prod(upscaling)
    return decode(upscaled_camera, detector)

In [ ]:
from mbloodmoon.mask import encode, decode

def dummy_sky_up(
    sky: np.array,
    base_camera: CodedMaskCamera,
    upscaled_camera: CodedMaskCamera,
) -> np.array:
    
    upscaling = UpscaleFactor(
        upscaled_camera.upscale_f.x - base_camera.upscale_f.x + 1,
        upscaled_camera.upscale_f.y - base_camera.upscale_f.y + 1,
    )
    
    # aperture = base_camera.mask.sum() / base_camera.mask.size
    # det_eff_response = base_camera.bulk.sum() / base_camera.bulk.size
    # counts_norm = 1 / (aperture * det_eff_response)
    counts_norm = 1

    encoded_det = counts_norm * encode(base_camera, sky)
    detector = _enlarge(encoded_det, upscaling) / np.prod(upscaling)
    norm = 1  # / adim effective area   # correction matrix due to source pos
    up_sky = norm * decode(upscaled_camera, detector)

    print(
        f"Base WFM detector shape: {base_camera.detector_shape}\n"
        f"Encoded detector shape: {encoded_det.shape}\n"
        f"Upscaled WFM detector shape: {upscaled_camera.detector_shape}\n"
        f"Upscaled encoded detector shape: {detector.shape}\n"
        f"Upscaled/Base WFM detector shape: {np.array(upscaled_camera.detector_shape) / np.array(base_camera.detector_shape)}\n\n"

        f"Base WFM sky shape: {base_camera.sky_shape}\n"
        f"Decoded sky shape: {sky.shape}\n"
        f"Upscaled WFM sky shape: {upscaled_camera.sky_shape}\n"
        f"Upscaled decoded sky shape: {up_sky.shape}\n\n"

        f"Sum encoded detector: {encoded_det.sum()}\n"
        f"Sum upscaled detector: {detector.sum()}\n"
        f"Min/Max sky: {sky.min(), sky.max()}\n"
        f"Sum sky: {sky.sum()}\n"
        f"Min/Max upscaled sky: {up_sky.min(), up_sky.max()}\n"
        f"Sum upscaled sky: {up_sky.sum()}\n"
    )
    return up_sky



mask_path = "/mnt/dbb8f47e-da06-47bf-8ef5-038092af70f7/Edos_Magnificent_Manor/PhD_AASS/Coding/IROS_Data/Simulations/wfm_mask.fits"
# mask_path = "/mnt/d/PhD_AASS/Coding/Images_fits/wfm_mask.fits"
wfm_base = codedmask(
    mask_filepath=mask_path,
    upscale_x=3,
    upscale_y=1,
)

n, m = wfm_base.sky_shape
y = n // 2 + n // 8
x = m // 2 - m // 8

sky = np.zeros((n, m)); sky[y, x] = 1e5

wfm_up = codedmask(
    mask_filepath=mask_path,
    upscale_x=3,
    upscale_y=7,
)


up_sky = dummy_sky_up(
    sky=sky,
    base_camera=wfm_base,
    upscaled_camera=wfm_up,
)

(650, 1040) (650, 3120) [0. 1.] 552240.0
(384, 1896) (384, 1896) [0. 1.] 552240.0
[0. 1.] 552240.0
n_zero_resp_pxs_y: 0
delta len det bins y: 0
n_zero_resp_pxs_x: 0
delta len det bins x: 0

(650, 1040) (4550, 3120) [0. 1.] 3865680.0
(2688, 1896) (2688, 1896) [0. 1.] 3865680.0
[0. 1.] 3853200.0
n_zero_resp_pxs_y: 8
delta len det bins y: -8
n_zero_resp_pxs_x: 0
delta len det bins x: 0

Base WFM detector shape: (384, 1896)
Encoded detector shape: (384, 1896)
Upscaled WFM detector shape: (2688, 1896)
Upscaled encoded detector shape: (2688, 1896)
Upscaled/Base WFM detector shape: [7. 1.]

Base WFM sky shape: (1033, 5015)
Decoded sky shape: (1033, 5015)
Upscaled WFM sky shape: (7237, 5015)
Upscaled decoded sky shape: (7237, 5015)

Sum encoded detector: 95913217364.63216
Sum upscaled detector: 95913217364.63213
Min/Max sky: (np.float64(0.0), np.float64(100000.0))
Sum sky: 100000.0
Min/Max upscaled sky: (np.float64(-3645240582.491308), np.float64(95888978188.31837))
Sum upscaled sky: -0.141500

In [20]:
up_sky[
    (wfm_up.upscale_f.y - wfm_base.upscale_f.y + 1) * y,
    (wfm_up.upscale_f.x - wfm_base.upscale_f.x + 1) * x,
]

np.float64(93655817879.85966)

In [ ]:
import numpy as np
from skimage.transform import resize
from scipy.ndimage import zoom

s = 10
a = np.arange(s * s).reshape(s, s)

upy, upx = 5, 5
a, resize(a, (upy, upx), order=2, mode="constant", preserve_range=True), resize(a, (upy, upx)).shape

(array([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
        [20, 21, 22, 23, 24, 25, 26, 27, 28, 29],
        [30, 31, 32, 33, 34, 35, 36, 37, 38, 39],
        [40, 41, 42, 43, 44, 45, 46, 47, 48, 49],
        [50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
        [60, 61, 62, 63, 64, 65, 66, 67, 68, 69],
        [70, 71, 72, 73, 74, 75, 76, 77, 78, 79],
        [80, 81, 82, 83, 84, 85, 86, 87, 88, 89],
        [90, 91, 92, 93, 94, 95, 96, 97, 98, 99]]),
 array([[ 5.51009193,  7.45692783,  9.5004175 , 11.5566443 , 13.929226  ],
        [26.0645532 , 27.51593906, 29.50112683, 31.52586652, 34.29929753],
        [46.53194303, 47.53371232, 49.50414803, 51.54095354, 54.76117092],
        [67.06171784, 67.61521363, 69.57353902, 71.62514108, 75.29646217],
        [89.70143267, 89.80437107, 91.79269645, 93.90408753, 98.12056674]]),
 (5, 5))

In [ ]:
import sys
import tempfile

import numpy as np
import numpy.typing as npt
from astropy.wcs import WCS
from astropy.wcs.utils import pixel_to_pixel
from astropy.wcs.wcsapi import SlicedLowLevelWCS

from reproject.array_utils import iterate_chunks, sample_array_edges
from reproject.utils import parse_input_data, parse_output_projection
from reproject.mosaicking.subset_array import ReprojectedArraySubset

IS_WIN = sys.platform == "win32"


def sky_composition(
    input_data,
    output_projection,
    shape_out,
    reproject_function,
    combine_function,
) -> tuple[npt.NDArray, npt.NDArray]:
    
    # Parse the output projection to avoid having to do it for each
    wcs_out, shape_out = parse_output_projection(output_projection, shape_out=shape_out)
    output_array = np.zeros(shape_out)
    output_footprint = np.zeros(shape_out)

    on_the_fly = combine_function in ("mean", "sum")


    # Start off by reprojecting individual images to the final projection
    if not on_the_fly:
        arrays = []

    with tempfile.TemporaryDirectory(ignore_cleanup_errors=IS_WIN) as local_tmp_dir:
        for idata in range(len(input_data)):
            # We need to pre-parse the data here since we need to figure out how to
            # optimize/minimize the size of each output tile (see below).
            array_in, wcs_in = parse_input_data(input_data[idata], hdu_in=None)

            # Since we might be reprojecting small images into a large mosaic we
            # want to make sure that for each image we reproject to an array with
            # minimal footprint. We therefore find the pixel coordinates of the
            # edges of the initial image and transform this to pixel coordinates in
            # the final image to figure out the final WCS and shape to reproject to
            # for each tile. We strike a balance between transforming only the
            # input-image corners, which is fast but can cause clipping in cases of
            # significant distortion (when the edges of the input image become
            # convex in the output projection), and transforming every edge pixel,
            # which provides a lot of redundant information.
            edges = sample_array_edges(array_in.shape, n_samples=11)[::-1]
            edges_out = pixel_to_pixel(wcs_in, wcs_out, *edges)[::-1]

            # Determine the cutout parameters

            # In some cases, images might not have valid coordinates in the corners,
            # such as all-sky images or full solar disk views. In this case we skip
            # this step and just use the full output WCS for reprojection.
            ndim_out = len(shape_out)
            if np.any(np.isnan(edges_out)):
                bounds = list(zip([0] * ndim_out, shape_out, strict=False))
            else:
                bounds = []
                for idim in range(ndim_out):
                    imin = max(0, int(np.floor(edges_out[idim].min() + 0.5)))
                    imax = min(shape_out[idim], int(np.ceil(edges_out[idim].max() + 0.5)))
                    bounds.append((imin, imax))
                    if imax < imin: break

            slice_out = tuple([slice(imin, imax) for (imin, imax) in bounds])

            if isinstance(wcs_out, WCS):
                wcs_out_indiv = wcs_out[slice_out]
            else:
                wcs_out_indiv = SlicedLowLevelWCS(wcs_out.low_level_wcs, slice_out)

            shape_out_indiv = tuple([imax - imin for (imin, imax) in bounds])

            array = footprint = None

            array, footprint = reproject_function(
                (array_in, wcs_in),
                output_projection=wcs_out_indiv,
                shape_out=shape_out_indiv,
                hdu_in=None,
                output_array=array,
                output_footprint=footprint,
            )

            # For the purposes of mosaicking, we mask out NaN values from the array
            # and set the footprint to 0 at these locations.
            reset = np.isnan(array)
            array[reset] = 0.0
            footprint[reset] = 0.0

            array = ReprojectedArraySubset(array, footprint, bounds)

            if on_the_fly:
                # By default, values outside of the footprint are set to NaN
                # but we set these to 0 here to avoid getting NaNs in the
                # means/sums.
                array.array[array.footprint == 0] = 0
                output_footprint[array.view_in_original_array] += array.footprint
                # We now need to do output[view] += array * footprint but to avoid
                # the temporary array allocation from array * footprint we modify
                # array inplace, which we can do as the array will be discarded at
                # the end of the loop.
                array.array *= array.footprint
                output_array[array.view_in_original_array] += array.array

            else:
                arrays.append(array)


        if combine_function == "mean":
            with np.errstate(invalid="ignore"):
                output_array /= output_footprint

        if combine_function in ("first", "last", "min", "max"):
            if combine_function == "min":
                output_array[...] = np.inf
            elif combine_function == "max":
                output_array[...] = -np.inf

            for array in arrays:
                if combine_function == "first":
                    mask = output_footprint[array.view_in_original_array] == 0
                elif combine_function == "last":
                    mask = array.footprint > 0
                elif combine_function == "min":
                    mask = (array.footprint > 0) & (
                        array.array < output_array[array.view_in_original_array]
                    )
                elif combine_function == "max":
                    mask = (array.footprint > 0) & (
                        array.array > output_array[array.view_in_original_array]
                    )

                output_footprint[array.view_in_original_array] = np.where(
                    mask, array.footprint, output_footprint[array.view_in_original_array]
                )
                output_array[array.view_in_original_array] = np.where(
                    mask, array.array, output_array[array.view_in_original_array]
                )

    # We need to avoid potentially large memory allocation from output == 0 so
    # we operate in chunks.
    for chunk in iterate_chunks(output_array.shape, max_chunk_size=256 * 1024**2):
        output_array[chunk][output_footprint[chunk] == 0] = 0

    return output_array, output_footprint